In [3]:
%load_ext autoreload
%autoreload 2

In [4]:
import _3_SchemaType as SchemaType
import importlib

importlib.reload(SchemaType)


<module '_3_SchemaType' from '/home/nakyung/projects/BDAIFin/5MODEL/STAGE2/_3_SchemaType.py'>

In [5]:
from _3_SchemaType import (
    build_stage2_dataset,
    ClientCardSchemaConfig,
)

cfg = ClientCardSchemaConfig(
    trans_path="../../5DATA/dataset/check_stage1",
    users_path="../../5DATA/original/users_data.csv",
    cards_path="../../5DATA/original/cards_data.csv",
)

stage2_df = build_stage2_dataset(cfg)

stage2_df.shape

/home/nakyung/projects/BDAIFin/5MODEL/STAGE2/_3_SchemaType.py:120: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["has_chip"] = df["has_chip"].replace({"YES": 1, "NO": 0}).astype("int8")


(166523, 55)

In [6]:
import pandas as pd 
df = pd.read_parquet("../../5DATA/dataset/check_stage1")
df["fraud"].value_counts()

fraud
0    166904
Name: count, dtype: int64

In [7]:
df.shape

(166904, 24)

In [8]:
stage2_df.to_parquet("../../5DATA/dataset/check_stage2")

In [9]:
stage2_df.isnull().sum()

id                          0
date                        0
client_id                   0
card_id                     0
amount                      0
merchant_id                 0
mcc                         0
is_online                   0
fraud                       0
has_error                   0
err_bad_card_number         0
err_bad_expiration          0
err_bad_cvv                 0
err_bad_pin                 0
err_bad_zipcode             0
err_insufficient_balance    0
err_technical_glitch        0
tx_year                     0
tx_month                    0
tx_day                      0
tx_hour                     0
weekday                     0
is_refund                   0
log_abs_amount              0
current_age                 0
per_capita_income           0
yearly_income               0
total_debt                  0
credit_score                0
num_credit_cards            0
has_chip                    0
num_cards_issued            0
credit_limit                0
year_pin_l

In [10]:
stage2_df.shape

(166523, 55)

In [11]:
from _5_ServingColumns import fit_artifacts, build_features_101, save_json, sanity_report
print("Imported OK:", fit_artifacts, build_features_101)

Imported OK: <function fit_artifacts at 0x7f7981990e50> <function build_features_101 at 0x7f7981991630>


In [12]:
from pathlib import Path
import pandas as pd
import shutil

HERE = Path.cwd()
ROOT = HERE
while ROOT.name != "BDAIFin" and ROOT != ROOT.parent:
    ROOT = ROOT.parent

if ROOT.name != "BDAIFin":
    raise RuntimeError(f"Cannot find BDAIFin root from cwd={HERE}")

DATA_ROOT = ROOT / "5DATA"
TRAIN_OUT = DATA_ROOT / "dataset" / "CHECK_STAGE2"
ART_PATH  = DATA_ROOT / "artifacts" / "stage2_check_artifacts.json"
# artifacts (train only)
MCC_STATS_PATH = DATA_ROOT / "artifacts" / "mcc_stats.parquet"

artifacts, mcc_stats_df = fit_artifacts(stage2_df)
save_json(artifacts, ART_PATH)
Path(MCC_STATS_PATH).parent.mkdir(parents=True, exist_ok=True)
mcc_stats_df.to_parquet(MCC_STATS_PATH, index=False)

print("saved artifacts:", ART_PATH)

# build features
df_train_feat = build_features_101(stage2_df, artifacts, mcc_stats_df)

print("train feat:", df_train_feat.shape)

sanity_report(df_train_feat)

p = Path(TRAIN_OUT)
if p.exists():
    shutil.rmtree(p)

df_train_feat.to_parquet(TRAIN_OUT, index=False)

print("saved CHECK_OUT:", TRAIN_OUT)

saved artifacts: /home/nakyung/projects/BDAIFin/5DATA/artifacts/stage2_check_artifacts.json
train feat: (166523, 65)
=== Sanity Check ===
n_features: 63
missing: []
extra: []
saved CHECK_OUT: /home/nakyung/projects/BDAIFin/5DATA/dataset/CHECK_STAGE2


In [13]:
df = pd.read_parquet(TRAIN_OUT)
df.shape

(166523, 65)